In [ ]:
# train_model.py

import os
import numpy as np
import cv2
from tensorflow.keras import layers, models

IMG_SIZE = 128

def load_data(base_dir):
    images, masks = [], []

    img_folder = os.path.join(base_dir, "images", "images", "train")
    mask_folder = os.path.join(base_dir, "masks", "masks", "train")

    for file in os.listdir(img_folder):

        if not file.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue

        img_path = os.path.join(img_folder, file)
        mask_path = os.path.join(mask_folder, file)

        img = cv2.imread(img_path, 0)
        mask = cv2.imread(mask_path, 0)

        if img is None or mask is None:
            continue

        # 🔥 SAR enhancement
        img = cv2.equalizeHist(img)

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0

        mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)[1]
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE)) / 255.0

        images.append(img)
        masks.append(mask)

    print("✅ Loaded:", len(images))

    return np.array(images).reshape(-1, IMG_SIZE, IMG_SIZE, 1), \
           np.array(masks).reshape(-1, IMG_SIZE, IMG_SIZE, 1)


# 🔥 STRONGER U-NET
def build_model():
    inputs = layers.Input((IMG_SIZE, IMG_SIZE, 1))

    c1 = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(32, 3, activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D()(c1)

    c2 = layers.Conv2D(64, 3, activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(64, 3, activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D()(c2)

    c3 = layers.Conv2D(128, 3, activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(128, 3, activation='relu', padding='same')(c3)

    u1 = layers.UpSampling2D()(c3)
    u1 = layers.concatenate([u1, c2])
    c4 = layers.Conv2D(64, 3, activation='relu', padding='same')(u1)

    u2 = layers.UpSampling2D()(c4)
    u2 = layers.concatenate([u2, c1])
    c5 = layers.Conv2D(32, 3, activation='relu', padding='same')(u2)

    outputs = layers.Conv2D(1, 1, activation='sigmoid')(c5)

    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy')

    return model


# ---- TRAIN ----
X, y = load_data("dataset")

model = build_model()

model.fit(X, y,
          epochs=30,
          batch_size=4,
          validation_split=0.1)

model.save("oil_spill_model.h5")

print("✅ Training complete!")

In [ ]:
# main_app
import cv2
import numpy as np
import tkinter as tk
from tkinter import filedialog, Label, Button, Canvas, Frame, ttk, Toplevel
from PIL import Image, ImageTk
import time
import winsound
from tensorflow.keras.models import load_model

# ---- Twilio ----
try:
    from twilio.rest import Client
    TWILIO_AVAILABLE = True
except:
    TWILIO_AVAILABLE = False

IMG_SIZE = 128

class OilResponseSim:
    def __init__(self, root):
        self.root = root
        self.root.title("SAR Oil Spill Analysis System")
        self.root.geometry("1100x850")
        self.root.configure(bg="#1e272e")

        self.model = load_model("oil_spill_model.h5")

        # ⚠️ REPLACE THESE WITH YOUR NEW TWILIO CREDENTIALS
        self.coast_guard_number = "+916363210252"
        self.twilio_sid = "AC1e5d9e7a2983f5b9768f1b8f22f5429f"
        self.twilio_auth = "e3229fab7d3c90881dd056ee8ea2bbfa"
        self.twilio_number = "+14783128651"

        self.current_spill_data = None
        self.call_made = False

        # ---- HEADER ----
        Label(root, text="🛰️ AI SAR OIL SPILL ANALYZER",
              font=("Arial", 22, "bold"),
              bg="#1e272e", fg="#00cec9").pack(pady=15)

        # ---- CANVAS ----
        self.canvas = Canvas(root, width=700, height=450,
                             bg="#2f3640", highlightthickness=2,
                             highlightbackground="#00cec9")
        self.canvas.pack(pady=10)

        # ---- STATUS ----
        self.status_lbl = Label(root, text="SYSTEM READY",
                                font=("Arial", 16, "bold"),
                                bg="#1e272e", fg="#00ff00")
        self.status_lbl.pack()

        self.class_lbl = Label(root,
                               text="Classification: Pending Scan",
                               font=("Arial", 12),
                               bg="#1e272e", fg="#f1c40f")
        self.class_lbl.pack()

        # ---- BUTTONS ----
        btn_frame = Frame(root, bg="#1e272e")
        btn_frame.pack(pady=10)

        Button(btn_frame, text="📂 Upload SAR Image",
               command=self.run_mission,
               bg="#0984e3", fg="white",
               font=("Arial", 12, "bold")).grid(row=0, column=0, padx=10)

        self.remedy_btn = Button(btn_frame, text="🛠️ Open Remediation Control",
                                command=self.open_remedy_window,
                                state="disabled",
                                bg="#27ae60", fg="white",
                                font=("Arial", 12, "bold"))
        self.remedy_btn.grid(row=0, column=1, padx=10)

        # ---- LOG ----
        self.log_box = Label(root, text="System Log:\n",
                             font=("Courier", 10),
                             bg="black", fg="#00ff00",
                             width=120, height=6,
                             anchor="nw", justify="left")
        self.log_box.pack(pady=15)

    # -------------------
    # LOGGING
    # -------------------
    def log(self, msg):
        current = self.log_box.cget("text")
        new = f"> {msg}\n" + "\n".join(current.split("\n")[:5])
        self.log_box.config(text=new)
        self.root.update()

    # -------------------
    # AI PREDICTION
    # -------------------
    def predict_mask(self, img):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray = cv2.equalizeHist(gray)

        resized = cv2.resize(gray, (IMG_SIZE, IMG_SIZE)) / 255.0
        input_img = resized.reshape(1, IMG_SIZE, IMG_SIZE, 1)

        pred = self.model.predict(input_img)[0]
        pred = (pred > 0.3).astype(np.uint8)

        mask = cv2.resize(pred, (img.shape[1], img.shape[0]))

        kernel = np.ones((7,7), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

        return mask

    # -------------------
    # CLASSIFICATION
    # -------------------
    def classify_oil(self, area):
        if area < 5000:
            return "Light Oil (Small)"
        elif area < 30000:
            return "Crude Oil (Medium)"
        else:
            return "Heavy Oil (Critical)"

    # -------------------
    # TWILIO ALERT 
    # -------------------
    def send_alert(self):
        if not TWILIO_AVAILABLE:
            self.log("Twilio not installed.")
            return

        try:
            client = Client(self.twilio_sid, self.twilio_auth)

            # 📞 CALL
            client.calls.create(
                twiml='<Response><Say voice="alice">Alert! Critical oil spill detected. Immediate action required.</Say></Response>',
                from_=self.twilio_number,
                to=self.coast_guard_number
            )

            # 📩 SMS
            client.messages.create(
                body="🚨 Critical Oil Spill Detected!",
                from_=self.twilio_number,
                to=self.coast_guard_number
            )

            self.log("📞 CALL + SMS sent successfully!")

        except Exception as e:
            self.log(f"Call failed: {e}")

    # -------------------
    # REMEDIATION WINDOW
    # -------------------
    def open_remedy_window(self):
        if not self.current_spill_data:
            return

        win = Toplevel(self.root)
        win.title("Remediation")
        win.geometry("500x400")
        win.configure(bg="#2d3436")

        Label(win, text="🛠️ REMEDIATION LOGISTICS",
              font=("Arial", 16, "bold"),
              bg="#2d3436", fg="#00cec9").pack(pady=20)

        data = self.current_spill_data

        Label(win, text=f"Type: {data['type']}\nArea: {data['area']:.2f}",
              bg="#2d3436", fg="white").pack()

        tank = ttk.Progressbar(win, length=300)
        tank.pack(pady=10)
        tank['value'] = 100

        def deploy():
            for i in range(10):
                tank['value'] -= 10
                win.update()
                time.sleep(0.3)

            Label(win, text="✅ Deployment Done",
                  fg="#00ff00", bg="#2d3436").pack()

        Button(win, text="Deploy",
               command=deploy,
               bg="#e67e22", fg="white").pack(pady=10)

    # -------------------
    # MAIN DETECTION
    # -------------------
    def run_mission(self):
        path = filedialog.askopenfilename()
        if not path:
            return

        self.canvas.delete("all")
        self.remedy_btn.config(state="disabled")

        self.log("Scanning image using AI model...")

        img = cv2.imread(path)
        mask = self.predict_mask(img)

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        total_area = 0

        for cnt in contours:
            area = cv2.contourArea(cnt)

            if area > 1500:
                total_area += area
                cv2.drawContours(img, [cnt], -1, (0, 255, 0), 2)

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        self.tk_img = ImageTk.PhotoImage(Image.fromarray(cv2.resize(img_rgb, (700, 450))))
        self.canvas.create_image(0, 0, anchor="nw", image=self.tk_img)

        if total_area > 0:
            oil_type = self.classify_oil(total_area)

            self.status_lbl.config(text="⚠ SPILL DETECTED", fg="#e74c3c")
            self.class_lbl.config(text=f"{oil_type} ({int(total_area)} px)")

            self.current_spill_data = {"type": oil_type, "area": total_area}
            self.remedy_btn.config(state="normal")

            self.log(f"Detected: {oil_type}")

            if "Critical" in oil_type:
                winsound.Beep(1000, 800)
                if not self.call_made:
                    self.send_alert()
                    self.call_made = True

        else:
            self.status_lbl.config(text="CLEAR", fg="#2ecc71")
            self.class_lbl.config(text="No Spill Detected")
            self.log("No spill detected.")


if __name__ == "__main__":
    root = tk.Tk()
    app = OilResponseSim(root)
    root.mainloop()